### Hyperparameter optimization with Optuna.

Default machine-learning -algorithm parameters are seldom the best possible ones, and we can usually improve the results by performing a hyperparameter optimization for the most promising canditate.

Most ML frameworks offer some sort of automated tooling for optimizing hyperparameters (e.g. *GridsearchCV* in sklearn), or you can manually loop over possible parameters

In [0]:
%run ../utils/run_target_helper

First import necessary libraries and data. Optuna is not included in the databricks ML runtime and should be configured manually from *compute->cluster->libraries*.

NOTE: install both *optuna* and *optuna-integration*

In [0]:
import optuna
from optuna.integration.mlflow import MLflowCallback

import mlflow
import mlflow.sklearn
from mlflow.models import infer_signature

from scipy.stats import page_trend_test
from sklearn.feature_extraction.text import TfidfVectorizer
from imblearn.ensemble import BalancedRandomForestClassifier
from sklearn.metrics import confusion_matrix, precision_score, recall_score

from pyspark.sql import SparkSession

import matplotlib.pyplot as plt
import seaborn as sns

# Disable autologging so we don't get extra runs.
mlflow.autolog(disable=True)

settings = get_settings(dbutils.widgets.get("TARGET"))
n_trials = int(dbutils.widgets.get("N_TRIALS"))
n_jobs = int(dbutils.widgets.get("N_JOBS"))
run_name = str(dbutils.widgets.get("RUN_NAME"))

gold_table = "verkkokauppa_reviews_gold" + settings["table_suffix"]
df_gold = spark.table(gold_table)

data = df_gold.select(
    "id", "lemmatized_text", "lemmatized_title", "positive_review", "train_test"
).toPandas()

In [0]:
to_tfidf = TfidfVectorizer(max_df=0.80, min_df=2, ngram_range=(1, 4))
tfidf_bag = to_tfidf.fit_transform(data["lemmatized_text"].values)

X_train = tfidf_bag.toarray()[data[data["train_test"] == "train"].index]
z_train = data[data["train_test"] == "train"]["positive_review"].values

X_test = tfidf_bag.toarray()[data[data["train_test"] == "test"].index]
z_test = data[data["train_test"] == "test"]["positive_review"].values

Optuna works by defining a study and a target function. The target function must define a parameter space for the parameters of interest, and return a score which can be used to track how good the parameters were.
The study will generate trials for the target function, and track the parameters which have already been tried.

Very simplistic example of finding the solution for function $$x^2 - 3x = 0$$:

In [0]:
example_study = optuna.create_study(direction="minimize")

def target_function(trial: optuna.Trial) -> float:
    x = trial.suggest_float("x", 0, 10)
    return abs(x**2 - 3 * x + 5)

example_study.optimize(target_function, n_trials=100)


After running the trials we can check the best found parameters.

In [0]:
print(example_study.best_trial)
print("Best found parameters: {}".format(example_study.best_trial.params))

In a more complex case it is best practice to define a class, which can handle all the non-optuna based initialization etc. in a separate function (usually *init*), and all the optuna-related things to the class *call* -function.

In [0]:
class OptunaObjective:
    def __init__(self, X_train, X_test, z_train, z_test):
        self.X_train = X_train
        self.X_test = X_test
        self.z_train = z_train
        self.z_test = z_test

    def set_mlflow_parent_run_id(self, parent_run_id: int):
        self.parent_run_id = parent_run_id

    def __call__(self, trial: optuna.Trial):
        n_estimators = trial.suggest_int("n_estimators", 10, 500, step=10)
        min_samples_split = trial.suggest_int("min_samples_split", 2, 20, step=2)
        min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10, step=1)
        max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])
        max_leaf_nodes = trial.suggest_categorical("max_leaf_nodes", [None, 4, 16, 64, 256, 1024, 4096])
        max_depth = trial.suggest_categorical("max_depth", [None, 4, 16, 64, 256, 1024, 4096])
        sampling_strategy = trial.suggest_categorical("sampling_strategy", ["all"])
        replacement = trial.suggest_categorical("replacement", [True])
        bootstrap = trial.suggest_categorical("bootstrap", [False])
        
        model = BalancedRandomForestClassifier(
            n_estimators=n_estimators,
            max_leaf_nodes=max_leaf_nodes,
            max_depth=max_depth,
            max_features=max_features,  # pyright: ignore
            min_samples_split=min_samples_split,
            min_samples_leaf=min_samples_leaf,
            # random_state=random_state,
            bootstrap=bootstrap,  # pyright: ignore
            replacement=replacement,  # pyright: ignore
            sampling_strategy=sampling_strategy,
        )

        #with mlflow.start_run(nested=True, parent_run_id=self.parent_run_id):
        model = model.fit(self.X_train, self.z_train)
        z_pred = model.predict(self.X_test)
        # print(confusion_matrix(z_test, z_pred))

        # Guard against models which give same label to all datapoints.
        # These might give best result if the data is imbalanced,
        # but are very uninformative.
        if (z_pred == 0).all() or (z_pred == 1).all():
            score = 0.0
            precision = 0.0
            recall = 0.0
        else:
            score = model.score(self.X_test, self.z_test)
            precision = precision_score(self.z_test, z_pred)
            recall = recall_score(self.z_test, z_pred)

        return score, precision, recall

Databricks MLflow integration does track the scikit-learn model runs automatically, but for better integration it is advised to create a callback funtion using optunas *MLFlowCallback*.

In [0]:

experiment_name = "/Users/jaakko.m.koivisto@gmail.com/reviews"
mlflow.set_experiment(experiment_name)
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id
print(experiment_name, experiment_id)

In [0]:
with mlflow.start_run(run_name=run_name, experiment_id=experiment_id) as parent_run:
    mlflow_callback = MLflowCallback(
        tracking_uri="databricks",
        metric_name=["accuracy", "precision", "recall"],
        create_experiment=False,
        mlflow_kwargs={"nested": True, "parent_run_id": parent_run.info.run_id},
    )

    study = optuna.create_study(directions=["maximize", "maximize", "maximize"])
    study.set_metric_names(["accuracy", "precision", "recall"])
    obj = OptunaObjective(X_train, X_test, z_train, z_test)
    obj.set_mlflow_parent_run_id(parent_run.info.run_id)
    import time
    start = time.time()
    study.optimize(obj, n_trials=n_trials, n_jobs=n_jobs, callbacks=[mlflow_callback])
    print(time.time() - start)

    study.trials.sort(key=lambda t: t.values[0])
    best_trial = study.trials[0]
                                   
    model_optimized = BalancedRandomForestClassifier(**best_trial.params)
    model_optimized = model_optimized.fit(X_train, z_train)
    z_pred_optimized = model_optimized.predict(X_test)

    mlflow.set_tag("Info", "Train model for review sentiment classification.")
    mlflow.set_tag("Model", "")

    mlflow.log_param("n_estimators", best_trial.params["n_estimators"])
    mlflow.log_param("min_samples_split", best_trial.params["min_samples_split"])
    mlflow.log_param("min_samples_leaf", best_trial.params["min_samples_leaf"])
    mlflow.log_param("max_features", best_trial.params["max_features"])
    mlflow.log_metric("precision", precision_score(z_test, z_pred_optimized))
    mlflow.log_metric("recall", recall_score(z_test, z_pred_optimized))
    mlflow.log_metric("accuracy", model_optimized.score(X_test, z_test))

    model_info = mlflow.sklearn.log_model(
        sk_model=model_optimized,
        artifact_path="review_sentiment_classifier",
        signature=infer_signature(X_train, z_pred_optimized),
        input_example=X_train,
        registered_model_name="verkkokauppa_review_classifier",
    )

In [0]:
sns.set_theme()

model_default = BalancedRandomForestClassifier(sampling_strategy="all", replacement=True, bootstrap=False)
model_default = model_default.fit(X_train, z_train)
z_pred = model_default.predict(X_test)

fig = plt.figure(figsize=(8, 8))
ax1 = fig.add_subplot(2, 2, 1)
ax2 = fig.add_subplot(2, 2, 2)

conf_mat_default = confusion_matrix(z_test, z_pred)
conf_mat_optimized = confusion_matrix(z_test, z_pred_optimized)

sns.heatmap(conf_mat_default, annot=True, fmt="d", cbar=False, cmap="flag", ax=ax1)
sns.heatmap(conf_mat_optimized, annot=True, fmt="d", cbar=False, cmap="flag", ax=ax2)
ax1.set_title("Default params.")
ax2.set_title("Optimized params.")

ax1.set_xlabel("Predicted")
ax1.set_ylabel("Actual")
ax2.set_xlabel("Predicted")